# Fetching data at warp speed

In [7]:
# 1. Install dependencies
!pip install dukascopy-python pandas pyarrow joblib -q

import os
import time
import pandas as pd
import dukascopy_python as dp
from datetime import datetime
from pathlib import Path
from joblib import Parallel, delayed

# --- CONFIGURATION ---
# NOTE: If you want to keep files permanently, mount Google Drive:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = Path("/content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ["EUR/USD"]
MONTHS = [
    "202401", "202402", "202403", "202404", "202405", "202406",
    "202407", "202408", "202409", "202410", "202411", "202412",
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512",
]

# PARALLELISM: Use -1 to use all cores, or 4-8 to avoid getting rate-limited
N_JOBS = 4 

# --- HELPER FUNCTIONS ---
def get_month_range(yyyymm: str):
    year, month = int(yyyymm[:4]), int(yyyymm[4:6])
    start = datetime(year, month, 1)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)
    return start, end

def process_job(pair, yyyymm, compression="snappy", max_retries=3):
    """Function to be executed in parallel with existence check and retries"""
    symbol = pair.replace("/", "").lower()
    
    # 1. EXISTENCE CHECK: Define file paths first
    out_name_bid = f"{symbol}_dukascopy_bid_{yyyymm}.parquet"
    out_name_ask = f"{symbol}_dukascopy_ask_{yyyymm}.parquet"
    out_path_bid = OUT_DIR / out_name_bid
    out_path_ask = OUT_DIR / out_name_ask

    # If both files already exist, skip the download completely
    if out_path_bid.exists() and out_path_ask.exists():
        return f"⏭️ Skipped (Already Exists): {pair} {yyyymm}"

    start, end = get_month_range(yyyymm)
    
    # 2. RETRY LOGIC: Handle Dukascopy connection drops
    for attempt in range(max_retries):
        try:
            df = dp.fetch(
                instrument=pair,
                interval=dp.INTERVAL_TICK,
                offer_side=dp.OFFER_SIDE_BID, 
                start=start,
                end=end,
            )

            if df is None or len(df) == 0:
                return f"⚠️ Empty: {pair} {yyyymm}"

            results_stats = []
            for side, price_col, vol_col in [("bid", "bidPrice", "bidVolume"), ("ask", "askPrice", "askVolume")]:
                out_name = out_name_bid if side == "bid" else out_name_ask
                out_path = out_path_bid if side == "bid" else out_path_ask
                
                pd.DataFrame({
                    "datetime": df.index,
                    "symbol": symbol.upper(),
                    "price_type": side.upper(),
                    "price": df[price_col].values,
                    "volume": df[vol_col].values,
                }).to_parquet(out_path, engine="pyarrow", compression=compression, index=False)
                
                results_stats.append(f"{out_name} ({len(df):,} rows)")
                
            return f"✅ Success: {pair} {yyyymm} -> " + " & ".join(results_stats)
        
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
                continue
            return f"❌ FAILED: {pair} {yyyymm} | Error after {max_retries} attempts: {str(e)}"

# --- EXECUTION ---
jobs = [(p, m) for p in PAIRS for m in MONTHS]
print(f"🚀 Starting Parallel Download of {len(jobs)} jobs using {N_JOBS} workers...\n")

t0 = time.time()
results = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(process_job)(p, m) for p, m in jobs
)

for r in results:
    print(r)

elapsed = time.time() - t0
print(f"\n✨ All done! Finished in {elapsed:.1f}s")
print(f"📁 Files saved to: {OUT_DIR.absolute()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Starting Parallel Download of 24 jobs using 4 workers...



INFO:DUKASCRIPT:current timestamp :2024-01-02T09:14:21.744000
INFO:DUKASCRIPT:current timestamp :2024-02-01T08:50:36.998000
INFO:DUKASCRIPT:current timestamp :2024-03-01T12:37:39.055000
INFO:DUKASCRIPT:current timestamp :2024-04-01T14:23:27.886000
INFO:DUKASCRIPT:current timestamp :2024-01-02T14:05:51.289000
INFO:DUKASCRIPT:current timestamp :2024-02-01T13:57:36.464000
INFO:DUKASCRIPT:current timestamp :2024-04-02T05:48:49.513000
INFO:DUKASCRIPT:current timestamp :2024-03-01T17:03:10.695000
INFO:DUKASCRIPT:current timestamp :2024-01-02T18:08:30.389000
INFO:DUKASCRIPT:current timestamp :2024-02-01T16:27:04.407000
INFO:DUKASCRIPT:current timestamp :2024-03-04T08:57:51.017000
INFO:DUKASCRIPT:current timestamp :2024-04-02T13:58:37.805000
INFO:DUKASCRIPT:current timestamp :2024-01-03T08:07:12.289000
INFO:DUKASCRIPT:current timestamp :2024-04-03T05:58:34.854000
INFO:DUKASCRIPT:current timestamp :2024-03-04T18:32:52.146000
INFO:DUKASCRIPT:current timestamp :2024-02-01T22:34:08.334000
INFO:DUK

✅ Success: EUR/USD 202401 -> eurusd_dukascopy_bid_202401.parquet (2,173,014 rows) & eurusd_dukascopy_ask_202401.parquet (2,173,014 rows)
✅ Success: EUR/USD 202402 -> eurusd_dukascopy_bid_202402.parquet (1,709,509 rows) & eurusd_dukascopy_ask_202402.parquet (1,709,509 rows)
✅ Success: EUR/USD 202403 -> eurusd_dukascopy_bid_202403.parquet (1,477,414 rows) & eurusd_dukascopy_ask_202403.parquet (1,477,414 rows)
✅ Success: EUR/USD 202404 -> eurusd_dukascopy_bid_202404.parquet (1,563,515 rows) & eurusd_dukascopy_ask_202404.parquet (1,563,515 rows)
✅ Success: EUR/USD 202405 -> eurusd_dukascopy_bid_202405.parquet (1,244,912 rows) & eurusd_dukascopy_ask_202405.parquet (1,244,912 rows)
✅ Success: EUR/USD 202406 -> eurusd_dukascopy_bid_202406.parquet (1,334,478 rows) & eurusd_dukascopy_ask_202406.parquet (1,334,478 rows)
✅ Success: EUR/USD 202407 -> eurusd_dukascopy_bid_202407.parquet (1,463,333 rows) & eurusd_dukascopy_ask_202407.parquet (1,463,333 rows)
✅ Success: EUR/USD 202408 -> eurusd_dukas